<a href="https://colab.research.google.com/github/Gajanana/Android-Clean-Boilerplate/blob/master/bge_small_en_v1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Local Inference on GPU
Model page: https://huggingface.co/BAAI/bge-small-en-v1.5

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/BAAI/bge-small-en-v1.5)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from sentence_transformers import SentenceTransformer , SentenceTransformerTrainer, losses

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# sentences = [
#     "The weather is lovely today.",
#     "It's so sunny outside!",
#     "He drove to the stadium."
# ]
# embeddings = model.encode(sentences)

# similarities = model.similarity(embeddings, embeddings)
# print(similarities.shape)
# [3, 3]

# Load your saved model for inference
fine_tuned_model = SentenceTransformer("my_finetuned_model")

# Generate embeddings
embeddings = fine_tuned_model.encode(["Your sample text here"])

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [21]:
from google.colab import userdata
from datasets import Dataset
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
userdata.get('HF_TOKEN')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# print(os.environ['HF_TOKEN'])
# 1. Load the Excel file
import pandas as pd
df = pd.read_excel('/content/SytheticQandA.xlsx')

# Assume the text data is in a column named 'anchor'
df['anchor'].head()

# Fill NaN values with empty strings in the relevant columns
df['anchor'] = df['anchor'].fillna('')
df['positive'] = df['positive'].fillna('')
df['negetive'] = df['negetive'].fillna('')

train_examples = []
for index, row in df.iterrows():
    train_examples.append(InputExample(texts=[row['anchor'], row['positive'], row['negetive']]))

# 3. Create DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.TripletLoss(model=model)
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of training steps as warmup

model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=num_epochs,
          warmup_steps=warmup_steps,
          output_path='/content/fine-tuned-triplet-model', # Path to save the fine-tuned model
          show_progress_bar=True
          )

# Save the model
model.save('/content/fine-tuned-triplet-model')
# trainer = SentenceTransformerTrainer(
#     model=model,
#     train_dataset=train_dataloader,
#     loss=train_loss,
# )
# trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

result = client.feature_extraction(
    "Today is a sunny day and I will get some ice cream.",
    model="BAAI/bge-small-en-v1.5",
)